# 00 - Concepts Primer

**Read this first.** Every term you will hear in Q&A is here, defined in plain English with the *role each one plays in our project*.

Companion notebooks:
- `01_phase1_resnet_baseline.ipynb` - Phase 1 (ResNet baseline)
- `02_phase2_vit_bugs_and_fixes.ipynb` - Phase 2 (ViT + the 3 bugs)
- `03_phase3_dinov2_foundation.ipynb` - Phase 3 (frozen DINOv2)
- `04_results_and_story.ipynb` - tables + the 4 insights to recite
- `PRESENTATION_FLOW.md` - slide-by-slide one-liners
- `QA_CHEATSHEET.md` - 20 likely instructor questions

## 1. The whole pipeline in one diagram

Every phase keeps the same outer skeleton. Only the **backbone** changes.

```mermaid
flowchart LR
    Frame[Video frame I_t] --> Backbone[Backbone f_theta]
    Backbone --> Bridge[Bridge / FPN]
    Bridge --> RPN[RPN proposes boxes]
    RPN --> RoI[RoI Align crops features per box]
    RoI --> DetHead[Detector head: object class + box refine]
    DetHead --> STTran[STTran spatio-temporal head]
    STTran --> Graph[Per-frame scene graph G_t]
```

- **Phase 1**: Backbone = ResNet-101.
- **Phase 2**: Backbone = plain ViT-B/16, Bridge = ViTDet synthetic pyramid.
- **Phase 3**: Backbone = **frozen** DINOv2 ViT-B/14, Bridge = DINOv2Bridge (same idea, different preprocessing source-of-truth).

Everything to the right of the bridge - RPN, RoI Align, detector heads, STTran - is **identical across phases**. That is the whole reason the comparison is clean.

## 2. Scene graph: static vs dynamic

**Scene graph (static):** for one image, a set of nodes (detected objects) and edges (relationships between them like `person - holding - cup`). Originated with the **Visual Genome** dataset (Krishna et al., 2017).

**Dynamic Scene Graph Generation (DSGG):** for every frame of a video, predict a *time-stamped* scene graph. Predicates can change over time (`person - looking_at - cup`, then `person - drinking_from - cup`).

**Why it matters here:** Action Genome's 26 relationship classes are split into three families - **attention**, **spatial**, **contact** - and each pair (person, object) can have *multiple* contact relations at once. So our relation head outputs are multi-label per family, which is why our relation loss is BCE not just CE.

## 3. Object detector anatomy: backbone / neck / head

Most detectors are split into three parts:

- **Backbone**: takes the raw image and produces a dense feature map. Examples: ResNet-101, ViT, DINOv2. *This is the part we keep replacing.*
- **Neck** (a.k.a. FPN, feature pyramid): re-organizes the backbone's features into multiple scales (`p2, p3, p4, p5`). Standard CNNs do this naturally; plain ViTs need a synthetic pyramid (this is what ViTDet provides).
- **Head**: small networks on top of pooled per-box features that produce class scores and box refinements.

**RPN (Region Proposal Network)** lives on top of the neck and proposes candidate boxes.
**RoI Align** crops backbone features inside each proposed box and pools them to a fixed 7x7 grid.

**Why it matters here:** Our entire project is one experiment - keep neck/RPN/RoI Align/head fixed, only swap the backbone. That's an ablation by design. Every gain we report is therefore attributable to the backbone.

## 4. Faster R-CNN (the detector skeleton we keep)

Two-stage detector by Ren et al., 2015. Stage 1: RPN proposes ~1000 boxes per image. Stage 2: RoI Align crops features per box, two heads produce (a) object class probabilities and (b) box-coordinate refinements.

**In our repo it lives at:** `STTran/lib/object_detector.py` line 157 instantiates `resnet(classes=..., num_layers=101, ...)`. Even in Phase 2/3 we keep the *same* `fasterRCNN` shell - we just feed it features from a different backbone via `_extract_base_features` (line 234).

**Why it matters here:** STTran's whole pipeline is built around "detector first, then graph head". If we threw out Faster R-CNN we would have to redesign everything. Our trick is: keep Faster R-CNN, swap only what feeds it.

## 5. ResNet-101

A 101-layer convolutional residual network (He et al., 2016). The original STTran backbone. Strong locally-aware features, but each conv only sees a small receptive field, so capturing long-range context ("the person in the corner is interacting with the cup on the table") requires going deep, which is expensive.

**Why it matters here:** This is the base STTran paper's choice and the Phase 1 baseline we reproduced to confirm parity (24.6 detector mAP@0.5).

## 6. Vision Transformer (ViT)

Dosovitskiy et al., 2021. Splits the image into **fixed-size patches** (16x16 pixels for ViT-B/16, 14x14 for the DINOv2 ViT-B/14), embeds each patch as a token, and runs a stack of transformer encoder blocks over the tokens. Every token attends to every other token, so the receptive field is global from layer 1.

Output for a 224x224 image is a 14x14 grid of patch tokens (= 196 tokens) plus a `[CLS]` prefix token.

**Why it matters here:**
- Phase 2 swaps the entire ResNet for a plain ViT (`vit_base_patch16_224` from `timm`).
- Phase 3 uses a DINOv2 ViT (`vit_base_patch14_dinov2`) and *freezes* it.
- The big architectural difference vs ResNet: ViTs produce a **single-scale** feature map (stride 16). Faster R-CNN expects a **multi-scale pyramid**. That mismatch is why we need ViTDet.

## 7. ViTDet - the synthetic feature pyramid

Li, Mao, Girshick, He, ECCV 2022. Trick: take a plain ViT's single-scale feature map at stride 16 and synthesize a 4-level pyramid by simple up/down sampling.

Concretely, given the ViT feature map `F` at stride 1/16:

- `p2`: ConvTranspose2d, stride 2 -> stride **1/8** (deeper resolution)
- `p3`: 1x1 Conv -> stride **1/16** (identity scale)
- `p4`: 3x3 Conv stride 2 -> stride **1/32**
- `p5`: 3x3 Conv stride 2 from p4 -> stride **1/64**

**In our repo:** see `phase2_vitdet/simple_vitdet_fpn.py` lines 81-98 where these four heads are defined, and lines 209-219 where they are applied in `forward()`.

**Why it matters here:** This is the bridge that makes a ViT "look like" a CNN backbone to the rest of Faster R-CNN. Without it, we would have to redesign RPN and RoI Align.

## 8. DINOv2 - self-supervised foundation model

Oquab et al., TMLR 2024 (Meta). A ViT trained with a self-supervised distillation objective on a curated 142M-image corpus, no labels. The result is a backbone whose **frozen** features are remarkably general-purpose: they transfer to classification, segmentation, depth estimation, retrieval - all by training only a tiny head on top.

Key practical traits:
- Patch tokens preserve fine spatial structure (the loss explicitly works at the patch level).
- Feature space already separates objects, parts, materials, etc., out of the box.
- Released variants: ViT-S/14, ViT-B/14, ViT-L/14, ViT-g/14. We use the B/14 variant.

**Why it matters here:** Phase 3 freezes DINOv2 (`requires_grad=False` on every backbone param) and uses it as a pure feature extractor. We only train the bridge, the detector neck/heads, and the STTran graph head.

## 9. DINOv3 + Gram anchoring (future work)

DINOv3 (Meta, 2024-2025) extends DINOv2 with **Gram anchoring**, a regularizer that constrains the patch-feature Gram matrix during training so that the geometric / textural relationships between patches are preserved. The practical effect is more discriminative per-patch features without sacrificing semantic transfer.

**Why it matters here:** It's our headline future-work direction, mentioned in the report's Conclusion. Even our DINOv2 result has a tiny residual gap to the base paper on PredCLS at certain recall thresholds; DINOv3 + Gram anchoring is the most direct candidate to close it.

## 10. STTran - the spatio-temporal graph head

Cong et al., ICCV 2021. Sits on top of the detector outputs. Given pair-features (subject, object, union-box, spatial mask, semantic embedding) for every (person, object) pair across a *temporal window*, it runs a transformer with two attention modes:

- **Spatial attention**: across all pairs *within the same frame*.
- **Temporal attention**: across the *same pair* over consecutive frames.

Then three small linear heads predict:
- attention relations (single-label, softmax/CE)
- spatial relations (multi-label, sigmoid/BCE)
- contacting relations (multi-label, sigmoid/BCE)

**In our repo:** `STTran/lib/sttran.py` lines 640-728. The transformer module is in `STTran/lib/transformer.py`.

**Why it matters here:** STTran is *unchanged* across our three phases. The only thing that changes is what feeds it. So when SGCLS jumps from 30 to 52, we know it is the backbone, not the relation head.

## 11. Encoder vs decoder - what the words mean inside STTran

STTran's transformer block has both an **encoder** (self-attention over pair features in a temporal window) and a **decoder** (cross-attention conditioning each pair on the encoder's output). They follow the standard Vaswani-style transformer roles: encoder builds context, decoder queries it.

**A subtle thing to know:** Our `ObjectClassifier` inside `STTran/lib/sttran.py` has its *own* small linear decoder (`decoder_lin` at line 107) that predicts object class from pooled RoI features. This is the source of the SGCLS "label_source" choice - we can use the classifier-decoder's prediction (`decoder`) or the upstream Faster R-CNN classifier head's prediction (`detector`). After the SGCLS fix, we default to `detector` which is more reliable on AG.

## 12. Action Genome dataset

Ji et al., CVPR 2020. Built on top of the Charades video dataset.

- **35 object classes**, **26 relationship classes** (paper says 25, codebase covers 26 with attention=3, spatial=6, contact=17).
- Three relation families: **attention** (e.g. `looking_at`), **spatial** (e.g. `in_front_of`), **contacting** (e.g. `holding`).
- After our runtime filtering: **167,068 valid training frames** from 7649 videos, **54,429 valid evaluation frames** from 1737 videos.

**Why it matters here:** Splits and class lists are loaded at the start of every training run, and the runtime filter (drop frames without a `person_bbox`) is what produces those exact numbers - they will be asked about.

## 13. The three evaluation modes (PredCLS / SGCLS / SGDET)

Standard scene-graph evaluation protocol. They differ in what is given to the model at evaluation time:

| Mode | Boxes given? | Object classes given? | Predict |
|---|---|---|---|
| **PredCLS** | ground-truth | ground-truth | only the *predicate* (relation) |
| **SGCLS** | ground-truth | NO - model classifies them | object class + predicate |
| **SGDET** | NO - model detects them | NO | everything (boxes + classes + predicates) |

Difficulty: PredCLS < SGCLS < SGDET. PredCLS isolates the relation head, SGCLS isolates the per-box embedding (because the localizer is bypassed), SGDET is the end-to-end task.

**Why it matters here:** Our headline win is on **SGCLS** specifically. The reason: SGCLS, with ground-truth boxes, directly probes per-region embedding quality, which is exactly where DINOv2's features dominate ViT-from-scratch.

**The Mixed-Path Bug story is anchored here.** The bug was that PredCLS and SGCLS used the *legacy ResNet path* while SGDET used the new ViT path - so the early ViT "results" were not actually a fair backbone comparison. Fixed in `STTran/lib/object_detector.py` line 767-770: now all three modes call `self._extract_base_features(...)` so they share one backbone.

## 14. Constraint regimes (With / Semi / No)

When evaluating recall, you have to decide how many predicates per (subject, object) pair count.

- **With constraint**: at most **one** predicate per ordered (subject, object) pair. Strictest.
- **Semi constraint**: top-k predicates per pair (we follow the STTran setting).
- **No constraint**: every (pair, predicate) pair is independently ranked. Loosest, recall is easy.

**Why it matters here:** Every result table has 3 modes x 3 constraints x 3 recall thresholds = 27 cells. The headline number people want is **SGCLS R@20 With-constraint** (most directly comparable to base STTran). That is the cell where we report 52.88 vs base 47.5.

## 15. Recall@K

For each test image, take the model's top-K predicted (subject, predicate, object) triplets. Recall@K = fraction of ground-truth triplets that appear among those top-K.

**Why recall, not mAP?** Scene-graph annotations are *incomplete* - the dataset cannot label every relation in every frame. Precision-based metrics would penalize the model for predicting a real relation that just wasn't annotated. Recall@K sidesteps that problem.

**Why R@10/20/50?** Three operating points across budgets: R@10 = a tight top-10 prediction list per image, R@50 = a generous top-50.

## 16. Mixed precision: BF16 vs FP32

**FP32**: 32-bit floating point. 8-bit exponent + 23-bit mantissa. Standard PyTorch default.
**BF16** (bfloat16): also 32-bit-style **8-bit exponent**, but only **7-bit mantissa**. Half the memory of FP32. Same dynamic range as FP32 but 16x less precision.

On H100 GPUs, BF16 is the default mixed-precision dtype (`torch.autocast(device_type='cuda', dtype=torch.bfloat16, ...)`).

**Why it matters here:** This is the **Poison Pill bug**. When the entropy term `-log(p)` inside cross-entropy gets a small probability `p`, the resulting log can need *more than 7 bits of mantissa* to be represented correctly. In BF16 it underflows or saturates, the loss returns `NaN/Inf`, and the entire batch is poisoned.

**The fix in our code:** `STTran/train.py` lines 997-999 explicitly cast the relation distributions to FP32 before computing CE/BCE:

```python
attention_distribution = pred['attention_distribution'].float()
spatial_distribution = pred['spatial_distribution'].float()
contact_distribution = pred['contacting_distribution'].float()
```

Backbone forward stays in BF16 (fast). Loss math runs in FP32 (stable). Best of both.

## 17. BatchNorm vs GroupNorm (and the 209-key bug)

**BatchNorm (BN)**: normalizes each channel using statistics over the *batch* dimension. Stores running mean and variance buffers.
**GroupNorm (GN)**: normalizes each channel using statistics over a *group of channels within one sample*. No running buffers. Behaves identically at train and eval time.

**Why it matters here:** Faster R-CNN ships with BatchNorm. ViT-based detectors typically use GroupNorm because BN behaves badly with very small effective batch sizes (which is what RoI Align produces). Mixing the two caused **209 detector-state-dict keys** (every BN `running_mean`/`running_var` pair) to be silently dropped at eval-load time, which made detector mAP collapse to zero.

**The fix in our code:** `STTran/lib/object_detector.py` lines 32-50 define `_replace_batchnorm_with_groupnorm`, applied at lines 187-206 conditional on `DETECTOR_BN_MODE=groupnorm`. The launch scripts now default this to `groupnorm` for ViT and DINOv2 phases (see `scripts/train_phase3_dinov2_h100.sbatch:91`).

## 18. Stage 1 vs Stage 2 (paper-faithful detector-first flow)

STTran's training is two-stage:

- **Stage 1**: train *only the detector*. Loss = object classification + bbox regression + RPN losses. Pick the best detector by `mAP@0.5`. (Code: `STTran/train_detector_stage1.py`.)
- **Stage 2**: freeze the detector. Train the STTran graph head end-to-end. Loss = object loss (kept light) + relation losses for attention/spatial/contact. (Code: `STTran/train.py`.)

**Sub-bug worth knowing:** the original Stage 1 still computed relation losses and skipped batches when those relation terms went `NaN`, even when the object loss itself was fine. Result: object pretraining was silently poisoned. Fix: Stage 1 was rewritten so `lambda_rel = 0`, i.e. relation losses are zeroed during Stage 1 (`STTran/train.py:990`).

## 19. The three-phase plan (the spine of our story)

From the original proposal (`knowledge information/st_project_proposal.pdf`):

1. **Phase 1**: Reproduce the original STTran baseline (ResNet-101 backbone) on Action Genome. Confirm we hit the published 71.8% PredCLS R@20 / 24.6 mAP numbers.
2. **Phase 2**: Replace the backbone with a fine-tuned plain ViT-Large. Hypothesis: global self-attention beats local convs for relationship detection. (Reality: this phase exposed three serious bugs first; see notebook 02.)
3. **Phase 3**: Use a frozen foundation model (DINOv2) as the backbone. Hypothesis: open-world generalized features can outperform task-specific fine-tuning. (Reality: yes, by a lot.)

**This phase plan is the spine of the presentation.** When in doubt during Q&A, retreat to it: "in Phase X, we did Y, and the result was Z."

## 20. The four insights to recite without looking

If you only remember four numerical things tomorrow, remember these:

1. **SGCLS R@20 (With) = 52.88 vs base STTran 47.5.** Headline win.
2. **SGDET R@20 (With) = 40.41 vs base 34.1.** End-to-end task win.
3. **Detector mAP@0.5: ResNet 24.6, ViT 23.19, DINOv2 24.25.** Detector quality is essentially the same across backbones - so the 22-point SGCLS jump is not from the localizer. It is from the *embedding*.
4. **PredCLS R@20 (With) = 71.71 (DINOv2) vs base 71.8.** Essentially tied, expected, because PredCLS isolates the relation head and we did not modify it.

These four together tell the whole story: the *backbone* is what carries the win, and the win shows up exactly where backbone embedding quality matters most (SGCLS), not where localization matters most (SGDET) or where everything is given (PredCLS).

---
**Next:** open `01_phase1_resnet_baseline.ipynb` to see how the ResNet baseline was reproduced.